# Efficient Frontier Portfolio Optimizer — Modern Portfolio Theory in Practice

**Author:** Jaik Wolfe  
**Type:** Intermediate-level quantitative finance project (Google Colab)

This notebook builds a real Modern Portfolio Theory optimizer: it pulls live multi-asset price history, estimates a shrinkage-adjusted covariance matrix, traces the full **efficient frontier**, solves for the **max-Sharpe** and **min-variance** portfolios, cross-checks the optimizer against a hand-built convex program, and — critically — tests whether any of it actually held up **out-of-sample**.

**How to use it:** set `TICKERS` in the Configuration cell below, then Runtime → Run all.

## Project Overview

Given a list of asset tickers, this notebook automatically:
1. Pulls 5 years of live daily price history for a multi-asset-class universe from Yahoo Finance.
2. Estimates annualized expected returns and a Ledoit-Wolf shrinkage-adjusted covariance matrix.
3. Solves for the max-Sharpe and minimum-variance portfolios using PyPortfolioOpt.
4. **Cross-checks** those results against a hand-written convex optimization (cvxpy) formulation.
5. Traces the full efficient frontier across a grid of target returns.
6. Runs an **out-of-sample test**: optimizes on the first 80% of history, holds the weights fixed, and sees how they actually performed on the held-out final 20% — compared to a naive equal-weight portfolio.
7. Runs a sensitivity analysis showing how much the "optimal" portfolio changes depending on the lookback window used to estimate returns.

## Real-World Finance Use Case

Mean-variance optimization is the textbook foundation of strategic asset allocation, and — with the estimation-error safeguards this notebook adds (shrinkage, sensitivity testing, out-of-sample validation) — it's also the practical foundation used at private banks, robo-advisors, and multi-asset funds to build model portfolios. The gap between the "textbook" Markowitz optimizer and what's actually usable in practice is almost entirely about handling estimation error, which is exactly what this notebook is built to demonstrate.

## System Architecture

```
┌────────────────────────────────────────┐    ┌──────────────────┐
│         Yahoo Finance (yfinance)         │    │    FRED (CSV)     │
│   multi-asset daily price history        │    │ (risk-free rate)  │
└─────────────────┬────────────────────────┘    └─────────┬─────────┘
                   ▼                                        ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │                    Data Layer (yfinance / requests)              │
   │        pull_prices()                    get_risk_free_rate()    │
   └───────────────────────────────┬───────────────────────────────-┘
                                    ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │              Estimation Layer (pandas / PyPortfolioOpt)          │
   │   expected_returns() → CovarianceShrinkage() (Ledoit-Wolf)       │
   └───────────────────────────────┬───────────────────────────────-┘
                                    ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │        Optimization Layer (PyPortfolioOpt / cvxpy cross-check)   │
   │   max_sharpe() · min_volatility() · efficient_return() (frontier)│
   │   manual_min_variance_qp() · manual_max_sharpe_qp()              │
   └───────────────────────────────┬───────────────────────────────-┘
                                    ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │           Validation Layer (out-of-sample + sensitivity)         │
   │   run_out_of_sample_test()        run_lookback_sensitivity()    │
   └───────────────────────────────┬───────────────────────────────-┘
                                    ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │              Presentation Layer (matplotlib / plotly)            │
   │   frontier chart · weight bars · OOS equity curve · sensitivity  │
   └─────────────────────────────────────────────────────────────────┘
```

## Required APIs and Data Sources

| Source | What it provides | Auth needed? |
|---|---|---|
| **Yahoo Finance** (`yfinance`) | 5 years of daily adjusted close prices across a multi-asset ETF universe | No key (unofficial wrapper — no SLA) |
| **FRED** (public CSV endpoint) | 10-Year Treasury yield used as the risk-free rate in Sharpe ratio calculations | No key needed |

## Required Python Libraries

`requests`, `pandas`, `numpy`, `yfinance`, `scipy`, `PyPortfolioOpt`, `cvxpy`, `matplotlib`, `plotly` — all pip-installable, installed in the first code cell.

## Folder/File Structure

```
Efficient_Frontier_Optimizer.ipynb
├── Cell: Setup & installs
├── Cell: Configuration (TICKERS, lookback, weight bounds, OOS split)
├── Section 1 — Data pipeline               (pull_prices, get_risk_free_rate)
├── Section 2 — Expected returns & risk      (mu, Ledoit-Wolf covariance)
├── Section 3 — Core optimization            (max_sharpe, min_volatility)
├── Section 4 — Manual cvxpy cross-check     (independent QP re-derivation)
├── Section 5 — Efficient frontier tracing   (grid of target returns)
├── Section 6 — Out-of-sample validation     (train/hold/test vs. equal-weight)
├── Section 7 — Sensitivity analysis         (lookback window sweep)
├── Section 8 — Visualizations               (4 charts)
└── Section 9 — Final summary report
```

## Setup

Install dependencies (safe to re-run).

In [ ]:
%pip install -q requests pandas numpy yfinance scipy PyPortfolioOpt cvxpy matplotlib plotly

## Configuration

**Edit `TICKERS` to whatever universe you want to optimize over.** The default is a diversified multi-asset-class set (US equities, developed/emerging international equities, US bonds, long Treasuries, gold, REITs) chosen so the optimizer has genuinely different risk/return profiles to trade off.

In [ ]:
import warnings

import requests
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
import yfinance as yf
from pypfopt import expected_returns, risk_models, EfficientFrontier

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ----- USER-EDITABLE CONFIG -----------------------------------------------
TICKERS = ["SPY", "EFA", "EEM", "AGG", "TLT", "GLD", "VNQ"]
# SPY = US equities, EFA = developed intl equities, EEM = emerging markets,
# AGG = US aggregate bonds, TLT = long-term Treasuries, GLD = gold, VNQ = REITs
LOOKBACK_YEARS = 5          # years of history used for the main (in-sample) estimation
WEIGHT_BOUNDS = (0, 1)      # long-only, no leverage
OOS_TEST_FRACTION = 0.2     # fraction of history held out for the out-of-sample test
SENSITIVITY_LOOKBACKS = [1, 2, 3, 5]  # years, for the lookback-window sensitivity sweep
# ----------------------------------------------------------------------------

if len(TICKERS) < 2:
    raise ValueError("Provide at least 2 tickers — a single-asset 'portfolio' has nothing to optimize.")
if len(set(TICKERS)) != len(TICKERS):
    raise ValueError(f"TICKERS contains duplicates: {TICKERS}")

print(f"Configured universe: {TICKERS}")

## Section 1 — Data Pipeline

Pulls daily adjusted close prices for the whole universe in one batched call, validates that every ticker actually returned data, and drops any dates with missing values (rather than silently forward-filling, which would understate risk).

In [ ]:
def pull_prices(tickers: list, years: float) -> pd.DataFrame:
    """Pull daily adjusted close prices for a ticker universe over the trailing N years."""
    raw = yf.download(tickers, period=f"{years}y", auto_adjust=True, progress=False)
    if raw.empty:
        raise RuntimeError(
            f"yfinance returned no data at all for {tickers}. Check the tickers are valid and that "
            "you have an internet connection."
        )
    prices = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].rename(columns={"Close": tickers[0]})

    missing_tickers = [t for t in tickers if t not in prices.columns or prices[t].dropna().empty]
    if missing_tickers:
        raise RuntimeError(f"No price data returned for: {missing_tickers}. Double-check these tickers.")

    before = len(prices)
    prices = prices.dropna()
    dropped = before - len(prices)
    if dropped:
        print(f"Dropped {dropped} row(s) with missing data across the universe (kept {len(prices)} aligned trading days).")
    if len(prices) < 252:
        raise RuntimeError(f"Only {len(prices)} trading days of clean overlapping history — too little to estimate risk reliably.")
    return prices


def get_risk_free_rate() -> float:
    """Latest 10-Year Treasury yield (DGS10) from FRED's public CSV endpoint, as a decimal."""
    resp = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10", timeout=20)
    resp.raise_for_status()
    rows = [r for r in resp.text.strip().split("\n")[1:] if r and r.split(",")[-1] != "."]
    if not rows:
        raise RuntimeError("FRED returned no usable DGS10 observations.")
    return float(rows[-1].split(",")[-1]) / 100


prices = pull_prices(TICKERS, LOOKBACK_YEARS)
risk_free_rate = get_risk_free_rate()
print(f"Pulled {len(prices)} trading days from {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Risk-free rate (10Y UST): {risk_free_rate:.2%}")
display(prices.tail())

## Section 2 — Expected Returns & Risk (Ledoit-Wolf Shrinkage)

Expected returns use simple annualized historical mean returns (the noisiest input in mean-variance optimization — flagged explicitly in the caveats section). The covariance matrix uses **Ledoit-Wolf shrinkage**, which pulls the raw sample covariance toward a more stable structured target — standard practice because raw sample covariance is notoriously unstable with a limited number of return observations relative to the number of assets.

In [ ]:
mu = expected_returns.mean_historical_return(prices)
S = risk_models.CovarianceShrinkage(prices).ledoit_wolf()

print("Annualized expected returns:")
display(mu.to_frame("expected_return").style.format("{:.2%}"))
print("\nAnnualized volatility (from the shrunk covariance diagonal):")
display((np.sqrt(np.diag(S)) ).round(4))

## Section 3 — Core Optimization: Max-Sharpe & Minimum-Variance

Solves for the two portfolios that matter most in practice: the **max-Sharpe** (tangency) portfolio and the **global minimum-variance** portfolio, both under a long-only, fully-invested constraint.

In [ ]:
def solve_max_sharpe(mu, S, bounds, rf):
    """Solve for the max-Sharpe (tangency) portfolio, with a clear error if the solver fails."""
    try:
        ef = EfficientFrontier(mu, S, weight_bounds=bounds)
        ef.max_sharpe(risk_free_rate=rf)
        return ef.clean_weights(), ef.portfolio_performance(risk_free_rate=rf)
    except Exception as e:
        raise RuntimeError(
            f"Max-Sharpe optimization failed ({type(e).__name__}: {e}). This can happen if weight_bounds "
            "are infeasible (e.g. bounds that can't sum to 1) or if S is not positive semi-definite."
        ) from e


def solve_min_volatility(mu, S, bounds):
    """Solve for the global minimum-variance portfolio."""
    try:
        ef = EfficientFrontier(mu, S, weight_bounds=bounds)
        ef.min_volatility()
        return ef.clean_weights(), ef.portfolio_performance()
    except Exception as e:
        raise RuntimeError(f"Min-volatility optimization failed ({type(e).__name__}: {e}).") from e


max_sharpe_weights, (ms_ret, ms_vol, ms_sharpe) = solve_max_sharpe(mu, S, WEIGHT_BOUNDS, risk_free_rate)
min_vol_weights, (mv_ret, mv_vol, mv_sharpe) = solve_min_volatility(mu, S, WEIGHT_BOUNDS)

print("MAX-SHARPE PORTFOLIO")
print(f"  Expected return: {ms_ret:.2%}   Volatility: {ms_vol:.2%}   Sharpe: {ms_sharpe:.2f}")
print("  Weights:", {k: round(v, 3) for k, v in max_sharpe_weights.items() if v > 0.001})
print()
print("MINIMUM-VARIANCE PORTFOLIO")
print(f"  Expected return: {mv_ret:.2%}   Volatility: {mv_vol:.2%}   Sharpe: {mv_sharpe:.2f}")
print("  Weights:", {k: round(v, 3) for k, v in min_vol_weights.items() if v > 0.001})

## Section 4 — Manual cvxpy Cross-Check

A library that silently gets the math wrong is worse than no library at all — so this section **independently re-derives both portfolios from scratch** using a raw `cvxpy` convex program, with no dependency on PyPortfolioOpt's internals. Minimum-variance is a straightforward quadratic program. Max-Sharpe is trickier (the raw Sharpe ratio isn't a convex objective), so this uses the standard **Cornuejols-Tütüncü change-of-variables trick**: introduce `y = kappa * w` and `kappa >= 0` so that maximizing the Sharpe ratio becomes an equivalent, solvable convex quadratic program.

In [ ]:
def manual_min_variance_qp(mu, S):
    """Independent cvxpy re-derivation of the global minimum-variance portfolio."""
    n = len(mu)
    w = cp.Variable(n)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, S.values)), [cp.sum(w) == 1, w >= 0])
    prob.solve()
    if w.value is None:
        raise RuntimeError(f"cvxpy min-variance solve failed (status: {prob.status}).")
    return pd.Series(np.clip(w.value, 0, None), index=mu.index)


def manual_max_sharpe_qp(mu, S, rf):
    """Independent cvxpy re-derivation of the max-Sharpe portfolio (Cornuejols-Tutuncu QP form)."""
    n = len(mu)
    y = cp.Variable(n)
    kappa = cp.Variable(nonneg=True)
    prob = cp.Problem(
        cp.Minimize(cp.quad_form(y, S.values)),
        [(mu.values - rf) @ y == 1, cp.sum(y) == kappa, y >= 0],
    )
    prob.solve()
    if y.value is None or kappa.value in (None, 0):
        raise RuntimeError(f"cvxpy max-Sharpe solve failed (status: {prob.status}).")
    w = np.clip(y.value / kappa.value, 0, None)
    return pd.Series(w / w.sum(), index=mu.index)  # renormalize away tiny numerical drift


manual_min_var_weights = manual_min_variance_qp(mu, S)
manual_max_sharpe_weights = manual_max_sharpe_qp(mu, S, risk_free_rate)

comparison = pd.DataFrame({
    "PyPortfolioOpt (max-Sharpe)": pd.Series(max_sharpe_weights),
    "Manual cvxpy (max-Sharpe)": manual_max_sharpe_weights,
    "PyPortfolioOpt (min-var)": pd.Series(min_vol_weights),
    "Manual cvxpy (min-var)": manual_min_var_weights,
})
max_abs_diff = (comparison["PyPortfolioOpt (max-Sharpe)"] - comparison["Manual cvxpy (max-Sharpe)"]).abs().max()
print(f"Largest weight discrepancy between PyPortfolioOpt and the manual QP: {max_abs_diff:.4f}")
if max_abs_diff > 0.01:
    print("WARNING: the two solvers disagree by more than 1 percentage point on some weight — investigate before trusting either.")
else:
    print("Cross-check passed: the independent cvxpy solve agrees with PyPortfolioOpt to within rounding error.")
display(comparison.style.format("{:.3f}"))

## Section 5 — Efficient Frontier Tracing

Solves `efficient_return()` across a grid of target returns to trace the full frontier — every portfolio here is "efficient" in the sense that no other long-only portfolio achieves the same return with less risk. Requesting a target return below the minimum-variance portfolio's own return is not an error — PyPortfolioOpt correctly returns the min-variance portfolio itself, since nothing on the frontier can go lower.

In [ ]:
def trace_efficient_frontier(mu, S, bounds, n_points=25):
    """Trace the efficient frontier by solving efficient_return() across a grid of target returns."""
    target_returns = np.linspace(mu.min(), mu.max() * 0.995, n_points)
    rows = []
    for target in target_returns:
        try:
            ef = EfficientFrontier(mu, S, weight_bounds=bounds)
            ef.efficient_return(target_return=target)
            ret, vol, sharpe = ef.portfolio_performance(risk_free_rate=risk_free_rate)
            rows.append({"target_return": target, "return": ret, "volatility": vol, "sharpe": sharpe})
        except (ValueError, cp.error.SolverError):
            continue  # infeasible corner of the grid — skip rather than crash the whole trace
    if not rows:
        raise RuntimeError("Could not trace any point on the efficient frontier — check mu/S for degeneracy.")
    return pd.DataFrame(rows).drop_duplicates(subset=["volatility"]).sort_values("volatility")


frontier = trace_efficient_frontier(mu, S, WEIGHT_BOUNDS)
print(f"Traced {len(frontier)} points on the efficient frontier.")
display(frontier.head())

## Section 6 — Out-of-Sample Validation

This is the step a lot of DIY "optimizers" skip, and it's the one that actually matters: optimize weights using only the first `1 - OOS_TEST_FRACTION` of history, **freeze those weights**, and see how they would have actually performed on the untouched remaining history — compared to a naive equal-weight portfolio that required no optimization at all.

In [ ]:
def run_out_of_sample_test(prices, bounds, rf, test_fraction):
    """Optimize on an in-sample window, hold the weights fixed, and evaluate them out-of-sample."""
    split_idx = int(len(prices) * (1 - test_fraction))
    train, test = prices.iloc[:split_idx], prices.iloc[split_idx:]
    if len(test) < 20:
        raise ValueError(f"Only {len(test)} out-of-sample days — increase OOS_TEST_FRACTION or LOOKBACK_YEARS.")

    train_mu = expected_returns.mean_historical_return(train)
    train_S = risk_models.CovarianceShrinkage(train).ledoit_wolf()
    ef = EfficientFrontier(train_mu, train_S, weight_bounds=bounds)
    ef.max_sharpe(risk_free_rate=rf)
    oos_weights = pd.Series(ef.clean_weights())

    test_returns = test.pct_change().dropna()
    equal_weights = pd.Series(1 / len(prices.columns), index=prices.columns)

    optimized_cum = (1 + (test_returns * oos_weights).sum(axis=1)).cumprod()
    equal_cum = (1 + (test_returns * equal_weights).sum(axis=1)).cumprod()

    return {
        "train_range": (train.index[0], train.index[-1]),
        "test_range": (test.index[0], test.index[-1]),
        "weights": oos_weights,
        "optimized_cum": optimized_cum,
        "equal_cum": equal_cum,
        "optimized_total_return": optimized_cum.iloc[-1] - 1,
        "equal_total_return": equal_cum.iloc[-1] - 1,
        "optimized_ann_vol": (test_returns * oos_weights).sum(axis=1).std() * np.sqrt(252),
        "equal_ann_vol": (test_returns * equal_weights).sum(axis=1).std() * np.sqrt(252),
    }


oos = run_out_of_sample_test(prices, WEIGHT_BOUNDS, risk_free_rate, OOS_TEST_FRACTION)
print(f"In-sample window:     {oos['train_range'][0].date()} to {oos['train_range'][1].date()}")
print(f"Out-of-sample window: {oos['test_range'][0].date()} to {oos['test_range'][1].date()}")
print(f"\nOptimized (in-sample max-Sharpe) weights held forward: {dict(oos['weights'].round(3))}")
print(f"\nOOS total return  — optimized: {oos['optimized_total_return']:+.2%}   equal-weight: {oos['equal_total_return']:+.2%}")
print(f"OOS annual volatility — optimized: {oos['optimized_ann_vol']:.2%}   equal-weight: {oos['equal_ann_vol']:.2%}")
if oos["optimized_ann_vol"] > oos["equal_ann_vol"] * 1.3:
    print("\nNote: the in-sample-optimized portfolio took on meaningfully MORE out-of-sample risk than "
          "equal-weight — a classic sign of overfitting to trailing historical returns.")

## Section 7 — Sensitivity Analysis: How Stable Is "Optimal"?

Re-runs the max-Sharpe optimization using only the trailing 1/2/3/5 years of history. If the "optimal" portfolio changes dramatically depending on an arbitrary lookback choice, that's not a quirk — it's the single biggest practical criticism of mean-variance optimization, and this makes it visible rather than hiding it.

In [ ]:
def run_lookback_sensitivity(all_prices, lookbacks, bounds, rf):
    """Re-solve max-Sharpe using only the trailing N years, for each N in `lookbacks`."""
    rows = []
    for years in lookbacks:
        cutoff = all_prices.index[-1] - pd.DateOffset(years=years)
        window = all_prices[all_prices.index >= cutoff]
        if len(window) < 60:
            print(f"Skipping {years}y lookback — only {len(window)} days available.")
            continue
        w_mu = expected_returns.mean_historical_return(window)
        w_S = risk_models.CovarianceShrinkage(window).ledoit_wolf()
        weights, (ret, vol, sharpe) = solve_max_sharpe(w_mu, w_S, bounds, rf)
        top_holdings = {k: round(v, 2) for k, v in weights.items() if v > 0.05}
        rows.append({"lookback_years": years, "return": ret, "volatility": vol, "sharpe": sharpe, "top_holdings": top_holdings})
    return pd.DataFrame(rows)


sensitivity = run_lookback_sensitivity(prices, SENSITIVITY_LOOKBACKS, WEIGHT_BOUNDS, risk_free_rate)
display(sensitivity.style.format({"return": "{:.2%}", "volatility": "{:.2%}", "sharpe": "{:.2f}"}))

## Section 8 — Visualizations

### 8.1 Efficient Frontier with Individual Assets & Capital Market Line

In [ ]:
asset_vols = np.sqrt(np.diag(S))
asset_rets = mu.values

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(frontier["volatility"], frontier["return"], color="#2563eb", linewidth=2, label="Efficient Frontier")
ax.scatter(asset_vols, asset_rets, color="#6b7280", s=60, zorder=5, label="Individual Assets")
for i, ticker in enumerate(mu.index):
    ax.annotate(ticker, (asset_vols[i], asset_rets[i]), textcoords="offset points", xytext=(6, 4), fontsize=9)

ax.scatter([ms_vol], [ms_ret], color="#dc2626", s=140, marker="*", zorder=6, label="Max-Sharpe Portfolio")
ax.scatter([mv_vol], [mv_ret], color="#16a34a", s=100, marker="D", zorder=6, label="Min-Variance Portfolio")

cml_x = np.linspace(0, frontier["volatility"].max() * 1.1, 50)
cml_y = risk_free_rate + (ms_ret - risk_free_rate) / ms_vol * cml_x
ax.plot(cml_x, cml_y, color="#dc2626", linestyle="--", linewidth=1.5, label="Capital Market Line")

ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlabel("Annualized Volatility")
ax.set_ylabel("Annualized Expected Return")
ax.set_title("Efficient Frontier — " + ", ".join(TICKERS), fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

### 8.2 Portfolio Weight Allocation Comparison

In [ ]:
weights_df = pd.DataFrame({
    "Max-Sharpe": pd.Series(max_sharpe_weights),
    "Min-Variance": pd.Series(min_vol_weights),
    "Equal-Weight": pd.Series(1 / len(TICKERS), index=TICKERS),
})

fig, ax = plt.subplots(figsize=(9, 5))
weights_df.plot(kind="bar", ax=ax, color=["#dc2626", "#16a34a", "#6b7280"])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel("Portfolio Weight")
ax.set_title("Weight Allocation by Strategy", fontsize=13, fontweight="bold")
ax.legend(title=None)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 8.3 Out-of-Sample Performance: Optimized vs. Equal-Weight

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(oos["optimized_cum"].index, (oos["optimized_cum"] - 1) * 100, color="#dc2626", linewidth=2, label="In-Sample-Optimized (held forward)")
ax.plot(oos["equal_cum"].index, (oos["equal_cum"] - 1) * 100, color="#6b7280", linewidth=2, label="Equal-Weight Benchmark")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Cumulative Return (%)")
ax.set_title("Out-of-Sample: Did the Optimization Actually Help?", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### 8.4 Interactive Sensitivity Chart — Lookback Window vs. Sharpe Ratio

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f"{y}y" for y in sensitivity["lookback_years"]],
    y=sensitivity["sharpe"],
    marker_color="#2563eb",
    text=sensitivity["top_holdings"].astype(str),
    hovertemplate="Lookback: %{x}<br>Sharpe: %{y:.2f}<br>Top holdings: %{text}<extra></extra>",
))
fig.update_layout(
    title="Max-Sharpe Ratio by Estimation Lookback Window (hover for holdings)",
    xaxis_title="Lookback Window", yaxis_title="Sharpe Ratio",
    width=750, height=450,
)
fig.show()

## Section 9 — Final Summary Report

In [ ]:
print("=" * 62)
print("  EFFICIENT FRONTIER OPTIMIZATION SUMMARY")
print("=" * 62)
print(f"  Universe: {', '.join(TICKERS)}")
print(f"  History: {prices.index[0].date()} to {prices.index[-1].date()} ({len(prices)} trading days)")
print(f"  Risk-free rate: {risk_free_rate:.2%}")
print("-" * 62)
print(f"  Max-Sharpe portfolio:   return {ms_ret:.2%}, vol {ms_vol:.2%}, Sharpe {ms_sharpe:.2f}")
print(f"  Min-variance portfolio: return {mv_ret:.2%}, vol {mv_vol:.2%}, Sharpe {mv_sharpe:.2f}")
print(f"  Manual cvxpy cross-check max weight discrepancy: {max_abs_diff:.4f}")
print("-" * 62)
print(f"  Out-of-sample optimized return:   {oos['optimized_total_return']:+.2%} (vol {oos['optimized_ann_vol']:.2%})")
print(f"  Out-of-sample equal-weight return: {oos['equal_total_return']:+.2%} (vol {oos['equal_ann_vol']:.2%})")
print("=" * 62)

## Performance Metrics

- **In-sample vs. out-of-sample spread** (Section 6) — the single most important number in this notebook. A large in-sample Sharpe ratio that evaporates or reverses out-of-sample is the signature of an overfit allocation, not a real edge.
- **Cross-check discrepancy** (Section 4) — how far the independent cvxpy solve is from PyPortfolioOpt; near-zero confirms the optimization itself is implemented correctly, independent of input quality.
- **Weight sensitivity across lookback windows** (Section 7) — how much the "optimal" allocation swings based on an arbitrary choice of estimation window; large swings mean the optimizer is picking up noise, not signal.

## Final Deliverables

- A reusable, universe-agnostic mean-variance optimizer (swap `TICKERS` and re-run).
- Max-Sharpe and minimum-variance portfolios, independently cross-checked against a hand-written QP.
- A full traced efficient frontier with the Capital Market Line and every asset plotted.
- An honest out-of-sample backtest against an equal-weight benchmark.
- A lookback-window sensitivity analysis exposing estimation-error risk.
- Four presentation-ready charts and a printed summary report.

## Resume Description

> *Built a mean-variance portfolio optimizer in Python using live multi-asset market data, Ledoit-Wolf shrinkage covariance estimation, and PyPortfolioOpt, independently validated the optimizer against a hand-derived convex program in cvxpy, and stress-tested the resulting allocations with an out-of-sample backtest and a lookback-window sensitivity analysis.*

## Potential Upgrades

- Add a Black-Litterman layer to blend market-implied equilibrium returns with your own views, instead of relying purely on noisy historical means.
- Add Hierarchical Risk Parity (HRP) as a third strategy alongside max-Sharpe and min-variance for comparison.
- Walk the out-of-sample test forward across multiple rolling windows instead of one single train/test split.
- Add sector/asset-class exposure constraints (e.g. cap any single asset at 30%) and see how the frontier shifts.
- Add real transaction costs and rebalancing frequency to the out-of-sample test.

## Modeling Caveats & Limitations

- **This is a teaching/portfolio model, not investment advice.** Section 6 and 7 exist specifically to show that mean-variance optimization is far less robust in practice than the textbook frontier chart suggests.
- Expected returns are estimated as simple trailing historical means — the noisiest possible input; real practitioners often use shrinkage estimators, factor models, or forward-looking capital market assumptions instead.
- `yfinance` is an unofficial, delayed data source with no SLA — fine for a learning project, not for anything time-sensitive or production-grade.
- The out-of-sample test uses a single train/test split rather than a rolling walk-forward — see "Potential Upgrades" for the more rigorous version.